In [2]:
# ============================================================
# IMPORTS
# ============================================================

import datetime
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import HalvingGridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [4]:

# ============================================================
# CONFIG
# ============================================================

warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")

INPUT_CSV = "../EDA/region_temp_extended.csv"
OUTPUT_DIR = Path("../Outputs/WaveletNN")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TODAY = datetime.datetime.today().strftime("%Y-%m-%d")
START = datetime.datetime.now()

RANDOM_STATE = 23
N_JOBS = -1
HORIZON = 1

DATE_COL = "date"
REGION_CANDIDATES = ["name", "region_code"]
TARGET_COL = "next_day"

FEATURE_COLS = [
    "dayofyear",
    "pdtn_doy",
    "de_trend_seas",
    "dts_doyavge",
    "dts_doyvar",
    "yday",
    "IIdays_ago",
    "IIIdays_ago",
    "IVdays_ago",
    "Vdays_ago",
    "VIdays_ago",
    "VIIdays_ago",
    "last_year",
    "last_2year",
    "last_3year",
    "last_4year",
    "last_5year",
    "h1_last_year",
    "h1_last_2years",
    "h1_last_3years",
    "h1_last_4years",
    "h1_last_5years",
    "h1_ly_2days",
    "h1_ly_3day",
    "h1_ly_4day",
    "h1_ly_5day",
    "h1_ly_6day",
    "h1_ly_7day",
    "h1_ly_next_1day",
    "h1_ly_next_2days",
    "h1_ly_next_3days",
    "h1_ly_next_4days",
    "h1_ly_next_5days",
    "h1_ly_next_6days",
    "h1_ly_next_7days",
    "diff_1year",
    "diff_2year",
    "diff_3year",
    "diff_4year",
    "diff_5year",
    "diff_yday",
    "diff_2days",
    "diff_3days",
    "diff_4days",
    "diff_5days",
    "diff_6days",
    "diff_7days",
    "last_7_1day_deltas_mean",
    "last_7_1day_deltas_min",
    "last_7_1day_deltas_max",
]

WNN_GRID = {
    "wnn__n_wavelets": [1, 2, 3, 5, 8, 12],
    "wnn__wavelet": ["mexican_hat", "morlet"],
    "wnn__alpha": [0.01, 0.1, 1.0, 10.0],
    "wnn__scale": [0.5, 1.0, 2.0],
    "wnn__center_strategy": ["quantile"],
    "wnn__include_original_features": [True, False],
}

OUTER_CV = TimeSeriesSplit(n_splits=5, test_size=365)
INNER_CV = TimeSeriesSplit(n_splits=3)


# ============================================================
# HELPERS
# ============================================================

def find_region_col(df: pd.DataFrame) -> str:
    for c in REGION_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"Could not find region column among {REGION_CANDIDATES}")


def create_next_day_target(group: pd.DataFrame) -> pd.DataFrame:
    g = group.copy().sort_values(DATE_COL)
    g[TARGET_COL] = g["de_trend_seas"].shift(-1)
    return g


def load_data() -> tuple[pd.DataFrame, str]:
    df = pd.read_csv(INPUT_CSV)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])

    region_col = find_region_col(df)

    required = {DATE_COL, region_col, *FEATURE_COLS}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df.sort_values([region_col, DATE_COL]).reset_index(drop=True)
    return df, region_col


def evaluate_fit(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    bias = np.mean(y_pred - y_true)
    r2 = r2_score(y_true, y_pred)
    return {
        "MAE": mae,
        "MAPE": mape,
        "Bias": bias,
        "R2": r2,
        "RMSE": rmse,
    }


# ============================================================
# WAVELET REGRESSOR
# ============================================================

class WaveletNNRegressor(BaseEstimator, RegressorMixin):
    """
    Sklearn-compatible WNN:
    - constructs wavelet basis functions from the inputs
    - fits a Ridge output layer on the transformed features
    """

    def __init__(
        self,
        n_wavelets=5,
        wavelet="mexican_hat",
        alpha=1.0,
        scale=1.0,
        center_strategy="quantile",
        include_original_features=True,
        random_state=23,
    ):
        self.n_wavelets = n_wavelets
        self.wavelet = wavelet
        self.alpha = alpha
        self.scale = scale
        self.center_strategy = center_strategy
        self.include_original_features = include_original_features
        self.random_state = random_state

    def _wavelet_fn(self, z):
        if self.wavelet == "mexican_hat":
            return (1.0 - z**2) * np.exp(-0.5 * z**2)
        elif self.wavelet == "morlet":
            return np.cos(1.75 * z) * np.exp(-0.5 * z**2)
        raise ValueError(f"Unsupported wavelet: {self.wavelet}")

    def _make_centers(self, X):
        n_features = X.shape[1]

        if self.center_strategy != "quantile":
            raise ValueError(f"Unsupported center strategy: {self.center_strategy}")

        quantiles = np.linspace(0.1, 0.9, self.n_wavelets)
        centers = np.zeros((self.n_wavelets, n_features), dtype=float)

        for j in range(n_features):
            centers[:, j] = np.quantile(X[:, j], quantiles)

        return centers

    def _transform(self, X):
        X = np.asarray(X, dtype=float)
        eps = 1e-8

        wavelet_features = []
        for k in range(self.n_wavelets):
            center = self.centers_[k]
            z = (X - center) / (self.scale_ + eps)
            phi = self._wavelet_fn(z)
            # aggregate over input dimensions -> one hidden unit output
            wavelet_features.append(phi.mean(axis=1))

        H = np.column_stack(wavelet_features)

        if self.include_original_features:
            H = np.column_stack([X, H])

        return H

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1)

        if X.shape[0] < 10:
            raise ValueError("Too few samples for WaveletNNRegressor.")

        self.centers_ = self._make_centers(X)
        self.scale_ = np.full(X.shape[1], float(self.scale), dtype=float)

        H = self._transform(X)

        self.output_model_ = Ridge(alpha=self.alpha, random_state=self.random_state)
        self.output_model_.fit(H, y)

        self.n_features_in_ = X.shape[1]
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        H = self._transform(X)
        return self.output_model_.predict(H)


# ============================================================
# PIPELINE
# ============================================================

def build_pipeline() -> Pipeline:
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer()),
            ("scaler", StandardScaler()),
            ("wnn", WaveletNNRegressor(random_state=RANDOM_STATE)),
        ]
    )


# ============================================================
# MAIN
# ============================================================

def main():
    feat_df, region_col = load_data()

    feat_df = (
        feat_df.groupby(region_col, group_keys=False)
        .apply(create_next_day_target)
        .reset_index(drop=True)
    )

    nested_cv_results = []
    best_params_dict = {}
    best_params_rows = []
    fit_metric_rows = []

    for region in feat_df[region_col].dropna().unique():
        print(f"\n=== REGION {region} ===")

        region_df = (
            feat_df.loc[
                feat_df[region_col] == region,
                [DATE_COL, region_col] + FEATURE_COLS + [TARGET_COL]
            ]
            .copy()
            .sort_values(DATE_COL)
        )

        train = region_df.dropna(axis=0, how="any").copy()

        if len(train) < 365 * 6:
            print(f"Skipping {region}: too few rows ({len(train)})")
            continue

        X = train[FEATURE_COLS]
        y = train[TARGET_COL]

        fold_results = []

        for fold_idx, (train_idx, test_idx) in enumerate(OUTER_CV.split(X), start=1):
            print(f"Outer Fold {fold_idx}")

            X_outer_train = X.iloc[train_idx]
            X_outer_test = X.iloc[test_idx]
            y_outer_train = y.iloc[train_idx]
            y_outer_test = y.iloc[test_idx]

            pipe = build_pipeline()

            grid_search = HalvingGridSearchCV(
                estimator=pipe,
                param_grid=WNN_GRID,
                cv=INNER_CV,
                scoring="neg_mean_absolute_percentage_error",
                refit=True,
                n_jobs=N_JOBS,
                factor=2,
                error_score="raise",
            )

            grid_search.fit(X_outer_train, y_outer_train)

            best_model = grid_search.best_estimator_
            y_pred = best_model.predict(X_outer_test)

            mape = mean_absolute_percentage_error(y_outer_test, y_pred)
            r2 = r2_score(y_outer_test, y_pred)

            row = {
                "model": "paper_wnn",
                "region": region,
                "mape": mape,
                "r2": r2,
                "horizon": HORIZON,
                "fold": fold_idx,
            }
            nested_cv_results.append(row)
            fold_results.append(row)

            fold_key = f"{region}_fold{fold_idx}"
            best_params_dict[fold_key] = {
                "model": "paper_wnn",
                "params": grid_search.best_params_,
            }

            fold_best_params = pd.DataFrame([grid_search.best_params_], index=[fold_key])
            fold_best_params["model"] = "paper_wnn"
            fold_best_params["region"] = region
            fold_best_params["fold"] = fold_idx
            best_params_rows.append(fold_best_params)

        nested_cv_df = pd.DataFrame(fold_results)
        if nested_cv_df.empty:
            continue

        min_row = nested_cv_df.loc[nested_cv_df["mape"].idxmin()]
        best_fold = f"{min_row['region']}_fold{int(min_row['fold'])}"
        best_params = best_params_dict[best_fold]["params"]

        final_pipe = build_pipeline()
        final_pipe.set_params(**best_params)
        final_pipe.fit(X, y)

        y_fit = final_pipe.predict(X)
        fit_metrics = evaluate_fit(y, y_fit)
        fit_metrics["Model"] = "paper_wnn"
        fit_metrics["region"] = region
        fit_metric_rows.append(fit_metrics)

        with open(OUTPUT_DIR / f"paper_wnn_pipeline_{region}.pkl", "wb") as f:
            pickle.dump(final_pipe, f)

    if best_params_rows:
        best_params_df = pd.concat(best_params_rows, axis=0)
        best_params_df.to_csv(OUTPUT_DIR / f"paper_wnn_best_params_{TODAY}.csv")

    if nested_cv_results:
        nested_cv_df_all = pd.DataFrame(nested_cv_results)
        nested_cv_df_all.to_csv(OUTPUT_DIR / f"paper_wnn_nested_cv_{TODAY}.csv", index=False)

    if fit_metric_rows:
        fit_metrics_df = pd.DataFrame(fit_metric_rows)
        fit_metrics_df.to_csv(OUTPUT_DIR / f"paper_wnn_fit_metrics_{TODAY}.csv", index=False)

    print("Time taken:", datetime.datetime.now() - START)


if __name__ == "__main__":
    main()


=== REGION 11 ===
Outer Fold 1


Python(24559) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(24562) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(24563) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(24564) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(24565) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(24566) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(24567) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(24568) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(24569) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(24570) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


TerminatedWorkerError: A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.

The exit codes of the workers are {SIGABRT(-6)}
Detailed tracebacks of the workers should have been printed to stderr in the executor process if faulthandler was not disabled.